<a href="https://colab.research.google.com/github/irajamuller/grover_adiabatico_experiments/blob/main/UA1_UA2_Algoritmo_de_Grover_Adiab%C3%A1tico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dwave-ocean-sdk --quiet
!pip install pyqubo --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.8/256.8 kB 5.2 MB/s eta 0:00:00


In [ ]:
import dimod
import dwave.samplers as samplers
import numpy as np
from collections import Counter
import time

In [ ]:
def construir_oracle_qubo(estado_alvo: str) -> dimod.BinaryQuadraticModel:
    """
    Constrói o BQM (Binary Quadratic Model) que representa o oráculo de
    Grover para o estado-alvo dado.

    Para cada qubit i com valor-alvo wᵢ ∈ {0, 1}:
      - Se wᵢ = 1: hᵢ = -1  → recompensa q_i = 1
      - Se wᵢ = 0: hᵢ = +1  → penaliza  q_i = 1

    Energia mínima = -n (apenas no estado-alvo); todos os outros estados
    têm energia maior, criando o poço de potencial do oráculo.

    Args:
        estado_alvo: string binária, ex: "11101"

    Returns:
        dimod.BinaryQuadraticModel no espaço BINARY
    """
    n = len(estado_alvo)
    linear = {}
    offset = 0.0

    for i, bit in enumerate(estado_alvo):
        wi = int(bit)
        # Equivalente a minimizar Σ (qᵢ - wᵢ)² expandido para QUBO binário
        linear[i] = 1.0 - 2.0 * wi   # coeficiente de qᵢ
        offset += wi                  # constante (deslocamento de energia)

    bqm = dimod.BinaryQuadraticModel(linear, {}, offset, dimod.BINARY)

    # Exibição Teórica
    print(f"\nEquação Hf: minimize H(q) = " + " + ".join([f"({linear[i]})*q_{i}" for i in range(n)]) + f" + {offset}")
    print(f"H0 (Teórico): -Σ σ_x_i (Superposição uniforme de todos os 2^{n} estados)")
    return bqm

def executar_grover_dwave(
    estado_alvo: str,
    num_leituras: int = 2000,
    verbose: bool = True
) -> tuple:
    """
    Executa o oráculo de Grover como problema QUBO no amostrador D-Wave
    Ocean (Simulated Annealing via dwave-samplers).

    O SA emula a dinâmica adiabática ao reduzir a temperatura gradualmente,
    favorecendo o estado de mínima energia (= estado-alvo).

    Para usar um QPU D-Wave real, substitua o amostrador por:
        from dwave.system import DWaveSampler, EmbeddingComposite
        sampler = EmbeddingComposite(DWaveSampler())

    Args:
        estado_alvo:  string binária alvo
        num_leituras: número de amostras independentes
        verbose:      imprimir tabela de resultados

    Returns:
        (response, lista_resultados_ordenados)
    """
    n = len(estado_alvo)
    bqm = construir_oracle_qubo(estado_alvo)

    # ── Amostrador: Simulated Annealing (emula annealing adiabático) ──────
    sampler = samplers.SimulatedAnnealingSampler()

    t0 = time.time()
    response = sampler.sample(
        bqm,
        num_reads=num_leituras,
        num_sweeps=2000,                 # "comprimento" do annealing
        beta_range=(0.05, 15.0),         # temperatura: quente → frio
        beta_schedule_type="geometric",  # resfriamento geométrico (padrão D-Wave)
        seed=42,
    )
    tempo_execucao = time.time() - t0

    # ── Coleta e análise dos resultados ───────────────────────────────────
    total_leituras = num_leituras

    resultados = []
    freq_estados = Counter()

    for amostra, energia, n_occ in response.data(["sample", "energy", "num_occurrences"]):
        estado_str = "".join(str(amostra[i]) for i in range(n))
        freq_estados[estado_str] += int(n_occ)

    for estado_str, contagem in sorted(freq_estados.items(),
                                        key=lambda x: -x[1]):
        energia = sum(
            bqm.linear[i] * int(estado_str[i]) for i in range(n)
        ) + bqm.offset
        prob = contagem / total_leituras
        resultados.append((estado_str, energia, contagem, prob))

    resultados.sort(key=lambda x: x[1])   # ordena por energia crescente

    # ── Impressão dos resultados ──────────────────────────────────────────
    if verbose:
        energia_alvo = -n + sum(int(b) for b in estado_alvo)  # energia mínima teórica
        print(f"\n  Tempo de execução : {tempo_execucao*1000:.1f} ms")
        print(f"  Leituras totais   : {total_leituras}")
        print(f"  Energia do alvo   : {float(bqm.linear[0])*0:.1f} "
              f"(mínimo global em |{estado_alvo}⟩)\n")

        colunas = f"  {'Estado':^{n+2}}  {'Energia':>8}  {'Amostras':>9}  {'P(estado)':>10}  "
        print(colunas + "Marcador")
        print("  " + "─" * (n + 2 + 8 + 9 + 10 + 20))

        for estado_str, energia, contagem, prob in resultados[:12]:
            marcador = "  ◄── ESTADO ALVO ✓" if estado_str == estado_alvo else ""
            bar = "▓" * int(prob * 25)
            print(f"  |{estado_str}⟩  {energia:>8.3f}  {contagem:>9}  "
                  f"{prob:>10.4%}  {bar}{marcador}")

        # Verifica se o alvo foi encontrado
        resultado_alvo = next((r for r in resultados if r[0] == estado_alvo), None)
        print()
        if resultado_alvo:
            _, e_alvo, c_alvo, p_alvo = resultado_alvo
            print(f"  ✅ Estado-alvo |{estado_alvo}⟩ ENCONTRADO")
            print(f"     Energia     : {e_alvo:.4f} (mínimo global confirmado)")
            print(f"     Ocorrências : {c_alvo} de {total_leituras} ({p_alvo:.2%})")
            print(f"     Rank         : #{resultados.index(resultado_alvo)+1} por energia")
        else:
            print(f"  ⚠️  Estado-alvo |{estado_alvo}⟩ não apareceu nas amostras")
            print(f"      (tente aumentar num_leituras ou num_sweeps)")

    return response, resultados


def executar_algoritmo(estado_alvo: str) -> None:
    """Executa todas as fases do algoritmo de Grover adiabático."""
    n = len(estado_alvo)
    N = 2 ** n
    sep = "═" * 70

    print(f"\n{'─'*70}")
    print(f"  [4/4] AMOSTRADOR D-WAVE (SimulatedAnnealingSampler)")
    print(f"{'─'*70}")
    print(f"\n  Oráculo QUBO enviado ao amostrador D-Wave Ocean...")
    num_leituras = 2000 if n <= 5 else 3000
    response, resultados = executar_grover_dwave(estado_alvo, num_leituras=num_leituras)

    # ── Resumo final ──────────────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"  RESUMO FINAL")
    print(f"{'─'*70}")
    resultado_alvo = next((r for r in resultados if r[0] == estado_alvo), None)
    if resultado_alvo:
        _, e_alvo, c_alvo, p_alvo = resultado_alvo
        print(f"  Estado-alvo   : |{estado_alvo}⟩")
        print(f"  Energia QUBO  : {e_alvo:.4f}  (mínimo global)")
        print(f"  Taxa de acerto: {p_alvo:.2%}  ({c_alvo}/{num_leituras} leituras)")
    print(f"\n{sep}\n")


if __name__ == "__main__":
    ESTADOS_ALVO = ["11101", "1011101"]

    print("\n" + "╔" + "═"*68 + "╗")
    print("║" + "  GROVER ADIABÁTICO — D-WAVE OCEAN SDK".center(68) + "║")
    print("║" + "  Roland & Cerf (2002) | QUBO / Ising Formulation".center(68) + "║")
    print("╚" + "═"*68 + "╝")

    for alvo in ESTADOS_ALVO:
        executar_algoritmo(alvo)


╔════════════════════════════════════════════════════════════════════╗
║                 GROVER ADIABÁTICO — D-WAVE OCEAN SDK               ║
║           Roland & Cerf (2002) | QUBO / Ising Formulation          ║
╚════════════════════════════════════════════════════════════════════╝

──────────────────────────────────────────────────────────────────────
  [4/4] AMOSTRADOR D-WAVE (SimulatedAnnealingSampler)
──────────────────────────────────────────────────────────────────────

  Oráculo QUBO enviado ao amostrador D-Wave Ocean...

Equação Hf: minimize H(q) = (-1.0)*q_0 + (-1.0)*q_1 + (-1.0)*q_2 + (1.0)*q_3 + (-1.0)*q_4 + 4.0
H0 (Teórico): -Σ σ_x_i (Superposição uniforme de todos os 2^5 estados)
